# 00 · Overview & setup — CFTR Variant Toolkit

A beginner-friendly, **honest** tour of the computational tools used to interpret
CFTR variants. Each notebook takes one tool (or tool family), explains *what it
is*, *what its score means*, *the decision threshold and why*, and *how to get the
real data* — then the archived integration notebook combines them into the missense-triage (A1) and splice-discordance (A2) worklists and notebook
08 covers the methodology pitfalls.

This notebook just checks your setup and shows the honest data-provenance map.

> **What are "A1" and "A2"?** The project has two analyses: **A1 — missense triage** (do single-amino-acid changes get mis-scored? → notebooks 01–08) and **A2 — splice discordance** (do non-coding / splice variants disrupt splicing, invisible to the missense tools? → notebooks 09–11). We use the plain terms "missense" and "splice" from here on.

In [1]:
import sys, pathlib
# `toolkit` is THIS repo's toolkit.py (one directory up) — NOT a pip
# package and nothing to do with gnomAD. The line below puts the repo
# root on sys.path so `import toolkit` resolves to ../toolkit.py.
sys.path.insert(0, str(pathlib.Path.cwd().parent))
import toolkit as tk
import pandas as pd, numpy as np
print("Python :", sys.version.split()[0])
print("toolkit:", pathlib.Path(tk.__file__).name)
# Print only the directory NAME (not the absolute machine path) + whether the
# gitignored cache is present. On a fresh clone this is MISSING — build/cache first.
print("cache  :", tk.CACHE_DIR.name + "/", "->",
      "FOUND" if tk.CACHE_DIR.exists() else "MISSING (fresh clone — see data/README.md)")


Python : 3.13.14
toolkit: toolkit.py
cache  : _tmp_fetch/ -> FOUND


## The one thing to understand before anything else: REAL vs DEMO

This toolkit is deliberately transparent about what is a *real* download/prediction
and what is a small *hand-curated demo table*. **But note: this repo ships NO data.**
`data/`, `outputs/`, and `_tmp_fetch/` are gitignored, so on a fresh clone every
loader either falls back to a DEMO table (with a warning) or raises
`FileNotFoundError` until you build/cache its extract. See **`../data/README.md`** for
how to fetch and build each one. **Never quote a DEMO number as a finding** — always
check the `source` column (the cell below prints it).

In [2]:
prov = pd.DataFrame([
  # tool, REAL once you..., coverage (once built), fresh-clone result, notebook
  ("gnomAD v4 (allele freq)", "cache _tmp_fetch/", "~2,466 missense / ~4,717 non-coding", "error", "01"),
  ("AlphaMissense",           "cache _tmp_fetch/", "genome-wide CFTR missense",           "error", "02"),
  ("ClinVar",                 "cache _tmp_fetch/", "genome-wide",                         "error", "07"),
  ("CFTR2 (30 Jan 2026)",     "build data/*.csv", "~2,097 variants / ~780 missense keys", "DEMO",  "08"),
  ("EVE",                     "build data/*.csv", "~26,809 CFTR variants",                "DEMO",  "03"),
  ("ESM1b",                   "build data/*.csv", "~28,120 CFTR (saturation)",            "DEMO",  "04"),
  ("REVEL",                   "build data/*.csv", "~10,826 CFTR (coord-keyed)",           "DEMO",  "05"),
  ("PrimateAI",               "build data/*.csv", "~1,976 CFTR (dbNSFP subset)",          "DEMO",  "06"),
  ("SpliceAI",                "build data/*.csv", "~566k CFTR SNVs (Illumina v1.3)",      "DEMO",  "09"),
  ("CADD",                    "live API",         "per-variant",                          "live",  "11"),
  ("Pangolin",                "n/a",              "9 curated splice variants",            "DEMO",  "10"),
], columns=["tool","real_when","coverage_once_built","fresh_clone","notebook"])
prov

,tool,real_when,coverage_once_built,fresh_clone,notebook
0,gnomAD v4 (allele freq),cache _tmp_fetch/,"~2,466 missense / ~4,717 non-coding",error,01
1,AlphaMissense,cache _tmp_fetch/,genome-wide CFTR missense,error,02
2,ClinVar,cache _tmp_fetch/,genome-wide,error,07
3,CFTR2 (30 Jan 2026),build data/*.csv,"~2,097 variants / ~780 missense keys",DEMO,08
4,EVE,build data/*.csv,"~26,809 CFTR variants",DEMO,03
5,ESM1b,build data/*.csv,"~28,120 CFTR (saturation)",DEMO,04
6,REVEL,build data/*.csv,"~10,826 CFTR (coord-keyed)",DEMO,05
7,PrimateAI,build data/*.csv,"~1,976 CFTR (dbNSFP subset)",DEMO,06
8,SpliceAI,build data/*.csv,~566k CFTR SNVs (Illumina v1.3),DEMO,09
9,CADD,live API,per-variant,live,11


In [3]:
# What do the loaders return HERE, right now? Print each one's ACTUAL source
# (REAL if its extract/cache exists, DEMO fallback otherwise) so the label can
# never disagree with the data. Cache-backed loaders raise if _tmp_fetch/ is empty.
def _src(df):
    return df["source"].iloc[0] if len(df) and "source" in df else "n/a"

def report(label, fn):
    try:
        df = fn()
        print(f"{label:16}: {len(df):>7} rows  source={_src(df)}")
    except FileNotFoundError as exc:
        print(f"{label:16}: MISSING  ({str(exc).splitlines()[0][:70]})")

for label, fn in [
    ("gnomAD missense", tk.load_gnomad_missense),
    ("AlphaMissense",   tk.load_alphamissense),
    ("ClinVar",         tk.load_clinvar),
    ("CFTR2",           tk.load_cftr2),
    ("EVE",             tk.load_eve),
    ("ESM1b",           tk.load_esm1b),
    ("REVEL",           tk.load_revel),
    ("PrimateAI",       tk.load_primateai),
    ("SpliceAI",        tk.load_spliceai),
]:
    report(label, fn)

gnomAD missense :    2466 rows  source=REAL
AlphaMissense   :    8597 rows  source=REAL
ClinVar         :    2801 rows  source=REAL
CFTR2           :    2097 rows  source=REAL
EVE             :   26809 rows  source=REAL
ESM1b           :   28120 rows  source=REAL
REVEL           :   10127 rows  source=REAL
PrimateAI       :    1976 rows  source=REAL


SpliceAI        :  566106 rows  source=REAL


## How to read this toolkit

One tool per notebook (03-11); the archived integration notebook is where they are combined and compared.

| Notebook | Tool | Data on a fresh clone |
|---|---|---|
| **01** | gnomAD — population allele frequency | REAL if cached, else error |
| **02** | AlphaMissense — genome-wide missense predictor | REAL if cached, else error |
| **03** | EVE — unsupervised evolutionary model | REAL if built, else DEMO |
| **04** | ESM1b — protein language model (backwards scale) | REAL if built, else DEMO |
| **05** | REVEL — supervised ensemble (+ circularity) | REAL if built, else DEMO |
| **06** | PrimateAI — semi-supervised (subset) | REAL if built, else DEMO |
| **07** | ClinVar — crowd-sourced clinical truth set | REAL if cached, else error |
| **08** | CFTR2 — functional truth set | REAL if built, else DEMO |
| **09** | SpliceAI — splice deltas (all CFTR SNVs) | REAL if built, else DEMO |
| **10** | Pangolin — independent splice model | DEMO always |
| **11** | CADD — live deleteriousness score | REAL (live API) |
| **12** | **Integration** — reproduce the missense + splice worklists & cross-tool comparisons | mixed (per loaders) |
| **13** | **Circular reasoning & training leakage** — the methodology | — |

Recommended order: 01 → 13 in sequence. If you only read two: **12** (what the
numbers really are, plus the tool-vs-tool comparisons) and **13** (why to be careful).

## Variant *consequence* — the vocabulary used throughout

A variant's **consequence** is what it does to the gene product, assigned by
mapping it onto the transcript (Ensembl VEP / Sequence Ontology terms; McLaren
et al. 2016, *Genome Biol*, PMID 27268795). The types this toolkit touches:

| Type | What it is | Which tools see it |
|---|---|---|
| **missense** | one amino acid swapped for another | AlphaMissense, EVE, ESM1b, REVEL, PrimateAI |
| **synonymous** | DNA changes, protein unchanged (can still break splicing) | splice tools, CADD |
| **inframe indel** | in-frame insertion/deletion of whole codons | (partially) missense tools |
| **splice donor / acceptor** | hits the canonical GT…AG splice signal | SpliceAI, Pangolin, CADD |
| **deep-intronic** | far inside an intron; can create a cryptic splice site | SpliceAI, Pangolin |
| **pLoF** (stop-gain / frameshift / splice) | predicted loss of function — truncates/abolishes the protein | (not scored by missense tools) |

**Missense tools are blind to splice / synonymous / non-coding variants** — which
is exactly why the project has a separate splice analysis (notebooks 09–11).

## To get REAL data instead of the DEMO fallback

Nothing ships in `data/` — see **`../data/README.md`** for the full fetch-and-build
guide. In short, each loader becomes REAL once you provide its extract:

- **Cache-backed** (need `_tmp_fetch/`, else `FileNotFoundError`): gnomAD (01),
  AlphaMissense (02), ClinVar (07).
- **Build-backed** (run the matching `build_*.py`, else DEMO fallback): CFTR2 (08),
  EVE (03), ESM1b (04), REVEL (05), PrimateAI (06), SpliceAI (09).
- **No data needed**: CADD (11, live API — cache it); Pangolin (10) is DEMO only.

The loaders set the `source` column for you; pass `strict=True` to raise instead of
silently falling back to DEMO. Join each on `protein_variant` (missense) or genomic
`chrom/pos/ref/alt` (splice/coord-keyed — mind CFTR's minus strand), then re-run
the archived integration notebook.